In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dosonleung/jd_comment_with_label/jd_comment_data.xlsx


In [20]:
import pandas as pd
import torch
import torch.nn as nn
import math
import time

from tqdm import tqdm
from dataclasses import dataclass
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from typing import List
from datasets import load_dataset, ClassLabel, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification



In [21]:
@dataclass
class Config:

    # 路径
    ROOT_DIR = Path('./').parent
    RAW_DATA_DIR = Path("/kaggle/input/datasets/dosonleung/jd_comment_with_label")
    PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"
    LOGS_DIR = ROOT_DIR / "logs"
    MODELS_DIR = ROOT_DIR / "models"

    # 训练参数
    SEQ_LEN = 128
    MAX_SEQ_LENGTH = 128
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-5
    EPOCHS = 20

    # 模型结构参数
    DIM_MODEL = 128
    NUM_HEADS = 4
    NUM_ENCODER_LAYERS = 2
    NUM_DECODER_LAYERS = 2

    MODEL_NAME = 'google-bert/bert-base-chinese'

    # 处理的取样的比例
    PROCESS_FRAC = 0.1

config = Config()

In [22]:
import os

if not os.path.exists(config.ROOT_DIR / 'models'):
    os.makedirs(config.ROOT_DIR / 'models')


if not os.path.exists(config.ROOT_DIR / 'data'):
    os.makedirs(config.ROOT_DIR / 'data')

if not os.path.exists(config.ROOT_DIR / 'data' / 'processed'):
    os.makedirs(config.ROOT_DIR / 'data' / 'processed')

In [23]:
df = pd.read_excel(config.RAW_DATA_DIR / "jd_comment_data.xlsx", usecols=['评价内容(content)', '评分（总分5分）(score)'])
dataset = Dataset.from_pandas(df)
dataset = dataset.rename_column('评价内容(content)', 'text').rename_column('评分（总分5分）(score)', 'labels')
dataset = dataset.cast_column("labels", ClassLabel(names=[0, 1, 2, 3, 4, 5]))

# 初始化Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
dataset = dataset.map(lambda x: tokenizer(x['text'], padding='max_length', truncation=True, max_length=128), batched=True).remove_columns(['text'])
dataset_dict = dataset.train_test_split(test_size=0.2, stratify_by_column="labels")


Casting the dataset:   0%|          | 0/71818 [00:00<?, ? examples/s]

Map:   0%|          | 0/71818 [00:00<?, ? examples/s]

In [24]:
dataset_dict.save_to_disk(config.PROCESSED_DATA_DIR)


Saving the dataset (0/1 shards):   0%|          | 0/57454 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14364 [00:00<?, ? examples/s]

In [25]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 57454
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14364
    })
})

In [26]:
from datasets import load_from_disk

def get_dataloader(train=True):
    path = config.PROCESSED_DATA_DIR / "train" if train else config.PROCESSED_DATA_DIR / "test"
    dataset = load_from_disk(path)
    dataset.set_format(type="torch")
    return DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

In [27]:
train_dataloader = get_dataloader(train=True)
test_dataloader = get_dataloader(train=False)
print(len(train_dataloader))
print(len(test_dataloader))
for batch in train_dataloader:
    for k, v in batch.items():
        print(k, v.shape)
    break

898
225
labels torch.Size([64])
input_ids torch.Size([64, 128])
token_type_ids torch.Size([64, 128])
attention_mask torch.Size([64, 128])


In [28]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)

        loss = outputs.loss
        total_loss += loss.item()

        # 反向传播
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
    return total_loss / len(dataloader)


def train():
    # 1. 设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # device = torch.device("cpu")
    
    # 2. 数据
    train_dataloader = get_dataloader(train=True)
    # 3. 模型
    model = AutoModelForSequenceClassification.from_pretrained(config.MODEL_NAME, num_labels=6).to(device)
    # 4. 优化器
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

    best_loss = float("inf")

    for epoch in range(1, config.EPOCHS + 1):
        print(f"========== Epoch {epoch} ==========")
        loss = train_one_epoch(model, train_dataloader, optimizer, device)
        print(f"Loss: {loss:.4f}")
        
        # 保存模型
        if loss < best_loss:
            best_loss = loss
            model.save_pretrained(config.MODELS_DIR)
            # torch.save(model.state_dict(), config.MODELS_DIR / "model.pt")

train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


========== Epoch 1 ==========


Training: 100%|██████████| 898/898 [18:51<00:00,  1.26s/it]

Loss: 0.2115


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

========== Epoch 2 ==========


Training:  16%|█▌        | 144/898 [03:01<15:52,  1.26s/it]


KeyboardInterrupt: 

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def predict_batch(model, input_ids):
    model.eval()
    with torch.no_grad():
        outputs = model(**input_ids)
        logits = outputs.logits

    return torch.argmax(logits, dim=1).tolist()


# def predict(text, model, tokenizer):
    
#     input_ids = tokenizer(text, padding='max_length', max_length=128, truncation=True, return_tensors='pt')
#     input_ids = {k: v.to(device) for k, v in input_ids.items()}
#     batch_result = predict_batch(model, input_ids)
    
#     return batch_result[0]

def eval_(model, dataloader, device):
    model.eval()
    total_cnt = 0
    correct_cnt = 0
    with torch.no_grad():
        for input_ids in tqdm(dataloader, '测试'):
            labels = input_ids.pop("labels").tolist()

            input_ids = {k: v.to(device) for k, v in input_ids.items()}

            batch_result = predict_batch(model, input_ids)

            for result, target in zip(batch_result, labels):
                if result == target:
                    correct_cnt += 1
                total_cnt += 1
    return correct_cnt / total_cnt
            

def evaluate():
    model = AutoModelForSequenceClassification.from_pretrained(config.MODELS_DIR).to(device)
    print("模型加载成功")
    test_dataloader = get_dataloader(train=False)

    acc = eval_(model, test_dataloader, device)
    print("acc: ", acc)



if __name__ == "__main__":
    evaluate()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

模型加载成功


测试: 100%|██████████| 225/225 [01:43<00:00,  2.18it/s]

acc:  0.9573934837092731
